# 01 — Random Forest Baseline

Train and evaluate a Random Forest on hand-crafted features.
This is the classical ML baseline — compare against the GRU in the next notebook.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, MODEL_CONFIG
from src.models import build_rf_pipeline
from src.metrics import (
    weighted_log_loss, macro_f1, per_class_metrics,
    plot_confusion_matrix, classification_summary,
)
from src.utils import encode_labels, get_class_name, plot_feature_importance, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
features = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'features_all.parquet'))
print(f"Feature table: {features.shape}")

## 1. Prepare Data

In [ ]:
# Separate features and labels
exclude_cols = ['object_id', 'target']
feature_cols = [c for c in features.columns if c not in exclude_cols]
X = features[feature_cols].values.astype(np.float32)
targets = features['target'].values

# Encode labels to 0..13
y, encode_map, decode_map = encode_labels(targets)
class_names_ordered = [get_class_name(decode_map[i]) for i in range(len(decode_map))]

print(f"Features: {X.shape}")
print(f"Labels: {y.shape}, classes: {np.unique(y)}")
print(f"NaN count: {np.isnan(X).sum()} ({100*np.isnan(X).sum()/X.size:.1f}%)")

In [ ]:
# Impute NaN with median
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)
print(f"After imputation — NaN count: {np.isnan(X).sum()}")

In [ ]:
# Stratified train/val/test split
rs = MODEL_CONFIG['random_state']
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=MODEL_CONFIG['test_size'], stratify=y, random_state=rs
)
relative_val = MODEL_CONFIG['val_size'] / (1 - MODEL_CONFIG['test_size'])
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=relative_val, stratify=y_trainval, random_state=rs
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

## 2. Train Random Forest

In [ ]:
rf = build_rf_pipeline()
rf.fit(X_train, y_train)
print("Training complete")

In [ ]:
# Cross-validation on train+val
cv_scores = cross_val_score(build_rf_pipeline(), X_trainval, y_trainval,
                            cv=5, scoring='f1_macro', n_jobs=-1)
print(f"5-fold CV macro F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

## 3. Evaluate on Test Set

In [ ]:
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)

results = classification_summary(y_test, y_pred, y_pred_proba,
                                 class_names={i: class_names_ordered[i]
                                              for i in range(len(class_names_ordered))})

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
plot_confusion_matrix(y_test, y_pred, class_names_ordered, normalize=True, ax=ax)
ax.set_title('Random Forest — Confusion Matrix (Normalized)')
plt.tight_layout()
save_figure(fig, 'rf_confusion_matrix')
plt.show()

## 4. Feature Importance

In [ ]:
# Extract importances from the RF inside the pipeline
importances = rf.named_steps['rf'].feature_importances_

fig, ax = plt.subplots(figsize=(10, 10))
plot_feature_importance(importances, feature_cols, top_n=30, ax=ax)
ax.set_title('Random Forest — Top 30 Feature Importances')
plt.tight_layout()
save_figure(fig, 'rf_feature_importance')
plt.show()

## 5. Save Model and Predictions

In [ ]:
os.makedirs('../../data', exist_ok=True)
joblib.dump(rf, '../../data/rf_model.pkl')
np.savez('../../data/rf_test_preds.npz',
         y_true=y_test, y_pred=y_pred, y_pred_proba=y_pred_proba,
         class_names=class_names_ordered)
print("Saved model and predictions")